In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import re
from sklearn.model_selection import train_test_split
import pandas as pd


In [ ]:


# Replace with your actual path
df = pd.read_csv("C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized/processed_dataset.csv")

In [19]:
df['filename'] = df['filename'].str.replace('.png', '', regex=False) + '_' + df['projection'].str.strip() + '.png'

In [21]:
df

,uid,filename,projection,findings,impression
0,1,1_IM-0001-4001.dcm_Frontal.png,Frontal,the cardiac silhouette and mediastinum size ar...,normal chest
1,1,1_IM-0001-3001.dcm_Lateral.png,Lateral,the cardiac silhouette and mediastinum size ar...,normal chest
2,2,2_IM-0652-1001.dcm_Frontal.png,Frontal,borderline cardiomegaly midline sternotomy enl...,no acute pulmonary findings
3,2,2_IM-0652-2001.dcm_Lateral.png,Lateral,borderline cardiomegaly midline sternotomy enl...,no acute pulmonary findings
4,3,3_IM-1384-1001.dcm_Frontal.png,Frontal,no information,no displaced rib fractures pneumothorax or ple...
...,...,...,...,...,...
13395,3489,3489_IM-1696-2001.dcm_Lateral.png,Lateral,the cardiomediastinal silhouette is normal in ...,negative for acute abnormality
13396,3490,3490_IM-1697-1001.dcm_Frontal.png,Frontal,no information,there is near spherical shaped calcification p...
13397,3490,3490_IM-1697-2001.dcm_Lateral.png,Lateral,no information,there is near spherical shaped calcification p...
13398,3491,3491_IM-1698-1001.dcm_Frontal.png,Frontal,the heart size and pulmonary vascularity appea...,prominent transverse aorta otherwise clear


In [22]:
# Remove the projection part from the filename to get a common ID
df['report_id'] = df['filename'].apply(lambda x: x.rsplit('_', 1)[0])

In [23]:
df

,uid,filename,projection,findings,impression,report_id
0,1,1_IM-0001-4001.dcm_Frontal.png,Frontal,the cardiac silhouette and mediastinum size ar...,normal chest,1_IM-0001-4001.dcm
1,1,1_IM-0001-3001.dcm_Lateral.png,Lateral,the cardiac silhouette and mediastinum size ar...,normal chest,1_IM-0001-3001.dcm
2,2,2_IM-0652-1001.dcm_Frontal.png,Frontal,borderline cardiomegaly midline sternotomy enl...,no acute pulmonary findings,2_IM-0652-1001.dcm
3,2,2_IM-0652-2001.dcm_Lateral.png,Lateral,borderline cardiomegaly midline sternotomy enl...,no acute pulmonary findings,2_IM-0652-2001.dcm
4,3,3_IM-1384-1001.dcm_Frontal.png,Frontal,no information,no displaced rib fractures pneumothorax or ple...,3_IM-1384-1001.dcm
...,...,...,...,...,...,...
13395,3489,3489_IM-1696-2001.dcm_Lateral.png,Lateral,the cardiomediastinal silhouette is normal in ...,negative for acute abnormality,3489_IM-1696-2001.dcm
13396,3490,3490_IM-1697-1001.dcm_Frontal.png,Frontal,no information,there is near spherical shaped calcification p...,3490_IM-1697-1001.dcm
13397,3490,3490_IM-1697-2001.dcm_Lateral.png,Lateral,no information,there is near spherical shaped calcification p...,3490_IM-1697-2001.dcm
13398,3491,3491_IM-1698-1001.dcm_Frontal.png,Frontal,the heart size and pulmonary vascularity appea...,prominent transverse aorta otherwise clear,3491_IM-1698-1001.dcm


In [25]:
df = df.drop(columns=["uid"], errors='ignore')


In [26]:
print(df.columns)


Index(['filename', 'projection', 'findings', 'impression', 'report_id'], dtype='object')


In [27]:
# Step 4: Keep only necessary columns
df_cleaned = df[['report_id', 'filename', 'findings', 'impression']]

# Step 5: Save to a new CSV file
df_cleaned.to_csv("cleaned_metadata.csv", index=False)

In [34]:
import os
import pandas as pd

# Load your metadata
df = pd.read_csv("C:/Users/ag331/OneDrive/Desktop/med_project/archive/cleaned_metadata.csv")

# Path to the folder where images are stored
image_dir = "C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized"  # replace with your actual folder path

# Loop through rows and rename files
for i, row in df.iterrows():
    # Original image name before renaming (e.g., without projection)
    base_name = row['filename'].rsplit('_', 1)[0] + '.png'  # 'IM-0652-1001.dcm.png'
    old_path = os.path.join(image_dir, base_name)

    # Target new name (with projection)
    new_path = os.path.join(image_dir, row['filename'])

    # Rename only if the file exists
    if os.path.exists(old_path):
        os.rename(old_path, new_path)
        print(f"✅ Renamed: {base_name} → {row['filename']}")
    else:
        print(f"❌ File not found for: {base_name}")

❌ File not found for: 1_IM-0001-4001.dcm.png
❌ File not found for: 1_IM-0001-3001.dcm.png
❌ File not found for: 2_IM-0652-1001.dcm.png
❌ File not found for: 2_IM-0652-2001.dcm.png
❌ File not found for: 3_IM-1384-1001.dcm.png
❌ File not found for: 3_IM-1384-2001.dcm.png
❌ File not found for: 4_IM-2050-1001.dcm.png
❌ File not found for: 4_IM-2050-2001.dcm.png
❌ File not found for: 5_IM-2117-1003002.dcm.png
❌ File not found for: 5_IM-2117-1004003.dcm.png
❌ File not found for: 6_IM-2192-1001.dcm.png
❌ File not found for: 6_IM-2192-2001.dcm.png
❌ File not found for: 7_IM-2263-1001.dcm.png
❌ File not found for: 7_IM-2263-2001.dcm.png
❌ File not found for: 8_IM-2333-1001.dcm.png
❌ File not found for: 8_IM-2333-2001.dcm.png
❌ File not found for: 9_IM-2407-1001.dcm.png
❌ File not found for: 9_IM-2407-2001.dcm.png
❌ File not found for: 10_IM-0002-2001.dcm.png
❌ File not found for: 10_IM-0002-1001.dcm.png
❌ File not found for: 11_IM-0067-1001.dcm.png
❌ File not found for: 11_IM-0067-2001.dcm.png


In [35]:
import os
import pandas as pd

df = pd.read_csv("C:/Users/ag331/OneDrive/Desktop/med_project/archive/cleaned_metadata.csv")
image_dir = "C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized"  # Replace with your image folder path

# 1. Show full folder path
print("Checking images in:", os.path.abspath(image_dir))

# 2. List all files in folder
available_files = set(os.listdir(image_dir))
print(f"\nTotal files found: {len(available_files)}")

# 3. Check each filename from CSV
missing = []
for fname in df['filename']:
    clean_fname = fname.strip()
    if clean_fname not in available_files:
        missing.append(clean_fname)

# 4. Report missing
if missing:
    print(f"\n❌ Missing {len(missing)} files:")
    for f in missing:
        print("   ", f)
else:
    print("\n✅ All filenames in CSV were found in the folder.")

Checking images in: C:\Users\ag331\OneDrive\Desktop\med_project\archive\_Images\_Images_normalized

Total files found: 7477

❌ Missing 824 files:
    44_IM-2078-1001.dcm_Lateral.png
    45_IM-2081-1001.dcm_Lateral.png
    60_IM-2192-1001.dcm_Lateral.png
    68_IM-2251-1001.dcm_Lateral.png
    71_IM-2273-1001.dcm_Lateral.png
    73_IM-2289-1001.dcm_Lateral.png
    74_IM-2296-2001.dcm_Lateral.png
    81_IM-2343-2001.dcm_Lateral.png
    88_IM-2394-2001.dcm_Lateral.png
    91_IM-2415-2001.dcm_Lateral.png
    104_IM-0031-0001-0002.dcm_Lateral.png
    119_IM-0128-1001.dcm_Lateral.png
    126_IM-0176-2002.dcm_Lateral.png
    130_IM-0198-2001.dcm_Lateral.png
    132_IM-0206-1001.dcm_Lateral.png
    134_IM-0219-1001.dcm_Lateral.png
    137_IM-0238-2002.dcm_Lateral.png
    144_IM-0283-1001.dcm_Lateral.png
    147_IM-0303-1001.dcm_Lateral.png
    157_IM-0372-1001.dcm_Lateral.png
    175_IM-0492-1001.dcm_Lateral.png
    178_IM-0509-2001.dcm_Lateral.png
    196_IM-0626-1001.dcm_Lateral.png
    198_

In [ ]:
import os

image_dir = "_Images/your_folder"  # change this to your actual path
print(sorted(os.listdir(image_dir)))

In [33]:
import os

image_dir = "_Images/your_folder"  # change this to your actual path
print(sorted(os.listdir(image_dir)))

FileNotFoundError: [WinError 3] The system cannot find the path specified: '_Images/your_folder'

## Tokenizer (Preserves Periods) + Vocabulary

In [20]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s\.]", "", text)  # ✅ keep periods for sentence boundaries
    text = re.sub(r"\d+", "", text)        # remove digits
    return text.strip().split()

def build_vocab(df, min_freq=2):
    df['report'] = df['findings'].fillna('') + ' ' + df['impression'].fillna('')
    tokenized = [tokenize(t) for t in df['report']]
    all_tokens = [token for line in tokenized for token in line]
    special_tokens = ["<pad>", "<start>", "<end>", "<unk>"]
    vocab = special_tokens + [w for w, c in Counter(all_tokens).items() if c >= min_freq]
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word


## Image Preprocessing

In [3]:
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


## Dataset for Paired Images + Reports

In [4]:
class CXRImagePairReportDataset(Dataset):
    def __init__(self, image_dir, dataframe, word2idx, transform=None, max_len=100):
        self.image_dir = image_dir
        self.df = dataframe
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filenames = row['filenames']
        if isinstance(filenames, str):
            filenames = filenames.strip("[]").replace("'", "").split(",")
        filenames = [f.strip() for f in filenames if f.strip()]
        if len(filenames) == 1:
            filenames *= 2
        elif len(filenames) > 2:
            filenames = filenames[:2]

        images = []
        for fname in filenames:
            path = os.path.join(self.image_dir, fname)
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise ValueError(f"Image missing: {path}")
            img = np.stack([img]*3, axis=-1)
            if self.transform:
                img = self.transform(img)
            images.append(img)

        image_tensor = torch.stack(images)  # [2, 3, 224, 224]

        report = (row.get('findings') or '') + ' ' + (row.get('impression') or '')
        tokens = ["<start>"] + tokenize(report)[:self.max_len] + ["<end>"]
        token_ids = [self.word2idx.get(t, self.word2idx["<unk>"]) for t in tokens]

        return image_tensor, torch.tensor(token_ids, dtype=torch.long)


## Collate Function

In [5]:
def collate_fn(batch):
    images, reports = zip(*batch)
    images = torch.stack(images)
    padded_reports = pad_sequence(reports, batch_first=False, padding_value=word2idx["<pad>"])
    return images, padded_reports


## Group Rows by Report ID

In [9]:
def group_report_rows(df):
    grouped_df = df.groupby('uid').agg({
        'filename': lambda x: list(x),
        'findings': 'first',
        'impression': 'first'
    }).reset_index()
    grouped_df = grouped_df.rename(columns={'filename': 'filenames'})
    return grouped_df


## Runner Function to Build DataLoaders

In [10]:
def prepare_data_and_loaders(csv_path, image_root_dir, batch_size=16):
    raw_df = pd.read_csv(csv_path)
    grouped_df = group_report_rows(raw_df)

    train_df, temp_df = train_test_split(grouped_df, test_size=0.25, random_state=42)
    valid_df, test_df = train_test_split(temp_df, test_size=0.4, random_state=42)

    global word2idx, idx2word
    word2idx, idx2word = build_vocab(train_df)

    train_dataset = CXRImagePairReportDataset(image_root_dir, train_df, word2idx, transform)
    valid_dataset = CXRImagePairReportDataset(image_root_dir, valid_df, word2idx, transform)
    test_dataset  = CXRImagePairReportDataset(image_root_dir, test_df, word2idx, transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    return train_loader, valid_loader, test_loader, word2idx, idx2word


## Run the Loader

In [11]:
# Set paths
csv_path = "C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized/train_split.csv"  # your metadata file
image_root = "_Images/_Images_normalized/div_images/train"  # image folder

# Prepare
train_loader, valid_loader, test_loader, word2idx, idx2word = prepare_data_and_loaders(
    csv_path=csv_path,
    image_root_dir=image_root,
    batch_size=16
)


## CHEXNET FOR FEATURE EXTRACTION

In [12]:
# 🟦 Cell 1: Load Pre-trained CheXNet (DenseNet-121)
import torch
import torch.nn as nn
from torchvision.models import densenet121

class CheXNetFeatureExtractor(nn.Module):
    def __init__(self):
        super(CheXNetFeatureExtractor, self).__init__()
        base_model = densenet121(pretrained=True)
        self.features = base_model.features  # up to the last conv block
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.output_dim = 1024  # DenseNet-121 last feature size

    def forward(self, x):
        batch_size, views, C, H, W = x.shape
        x = x.view(-1, C, H, W)  # flatten views
        out = self.features(x)
        out = self.avg_pool(out).view(out.size(0), -1)  # [batch*views, 1024]
        out = out.view(batch_size, views, -1)  # [batch, views, 1024]
        return out

chexnet = CheXNetFeatureExtractor()


c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## ENCODER

In [13]:
# 🟦 Cell 2: Transformer Encoder
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :].transpose(0, 1)
        return x

## FULL ARCH ENCODER, LSTM, & DECODER 

In [14]:
class ReportGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=3, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.chexnet = CheXNetFeatureExtractor()
        self.img_linear = nn.Linear(self.chexnet.output_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout),
            num_layers=num_layers
        )
        self.lstm = nn.LSTM(input_size=d_model, hidden_size=d_model, batch_first=True)
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_decoder = PositionalEncoding(d_model)
        self.transformer_decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout),
            num_layers=num_layers
        )
        self.out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, img_tensor, tgt, tgt_mask=None, tgt_key_padding_mask=None):
        img_feats = self.chexnet(img_tensor)
        img_feats = img_feats.mean(1)
        memory = self.img_linear(img_feats).unsqueeze(1)  # [batch, 1, d_model]
        memory = self.pos_encoder(memory.transpose(0, 1))
        memory = self.transformer_encoder(memory)
        memory, _ = self.lstm(memory.transpose(0, 1))
        memory = memory.transpose(0, 1)

        tgt_emb = self.embedding(tgt) * (self.d_model ** 0.5)
        tgt_emb = self.pos_decoder(tgt_emb)
        output = self.transformer_decoder(tgt_emb, memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_key_padding_mask)
        return self.out(output)

## Training Loop

In [15]:
import os
from torch.nn.utils import clip_grad_norm_
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction


def generate_square_subsequent_mask(sz):
    mask = torch.triu(torch.ones(sz, sz) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

def train_model(model, train_loader, valid_loader, optimizer, criterion, word2idx, idx2word, num_epochs=10, clip=1.0, save_path="best_model.pth"):
    best_bleu = 0
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, reports in train_loader:
            images, reports = images.to(device), reports.to(device)
            tgt_input = reports[:-1, :]
            tgt_output = reports[1:, :]

            tgt_mask = generate_square_subsequent_mask(tgt_input.size(0)).to(device)

            optimizer.zero_grad()
            output = model(images, tgt_input, tgt_mask=tgt_mask)
            loss = criterion(output.view(-1, output.size(-1)), tgt_output.view(-1))
            loss.backward()
            clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        bleu1, bleu2, bleu3, bleu4 = evaluate_bleu_scores(model, valid_loader, word2idx, idx2word)
        print(f"Epoch {epoch+1}, Train Loss: {avg_loss:.4f}, BLEU-1: {bleu1:.4f}, BLEU-2: {bleu2:.4f}, BLEU-3: {bleu3:.4f}, BLEU-4: {bleu4:.4f}")

        if bleu4 > best_bleu:
            best_bleu = bleu4
            torch.save(model.state_dict(), save_path)
            print("Best model saved.")

## Beam Search for Inference

In [16]:
def beam_search(model, img_tensor, word2idx, idx2word, max_len=50, beam_width=3):
    model.eval()
    device = next(model.parameters()).device
    start_token = torch.tensor([[word2idx['<start>']]], device=device)
    end_token = word2idx['<end>']
    img_tensor = img_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        img_feats = model.chexnet(img_tensor)
        img_feats = img_feats.mean(1)
        memory = model.img_linear(img_feats).unsqueeze(1)
        memory = model.pos_encoder(memory.transpose(0, 1))
        memory = model.transformer_encoder(memory)
        memory, _ = model.lstm(memory.transpose(0, 1))
        memory = memory.transpose(0, 1)

        sequences = [(start_token, 0.0)]
        for _ in range(max_len):
            all_candidates = []
            for seq, score in sequences:
                if seq[0, -1].item() == end_token:
                    all_candidates.append((seq, score))
                    continue
                tgt_emb = model.embedding(seq) * (model.d_model ** 0.5)
                tgt_emb = model.pos_decoder(tgt_emb)
                out = model.transformer_decoder(tgt_emb, memory)
                logits = model.out(out[-1, :])
                log_probs = torch.log_softmax(logits, dim=-1)
                topk = torch.topk(log_probs, beam_width)
                for i in range(beam_width):
                    next_token = topk.indices[i].unsqueeze(0).unsqueeze(0)
                    next_score = score + topk.values[i].item()
                    new_seq = torch.cat([seq, next_token], dim=1)
                    all_candidates.append((new_seq, next_score))
            sequences = sorted(all_candidates, key=lambda tup: tup[1], reverse=True)[:beam_width]
        best_seq = sequences[0][0].squeeze().tolist()
        return ' '.join([idx2word.get(tok, '<unk>') for tok in best_seq if tok != word2idx['<pad>']])


## Evaluation (BLEU)

In [17]:
def evaluate_bleu_scores(model, dataloader, word2idx, idx2word):
    model.eval()
    references = []
    hypotheses = []
    smooth = SmoothingFunction().method4
    with torch.no_grad():
        for images, reports in dataloader:
            for i in range(images.size(0)):
                img = images[i]
                ref = reports[1:, i].tolist()
                ref_words = [idx2word.get(tok, '<unk>') for tok in ref if tok not in (word2idx['<pad>'], word2idx['<end>'])]
                hyp = beam_search(model, img, word2idx, idx2word).split()
                references.append([ref_words])
                hypotheses.append(hyp)
    bleu1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smooth)
    bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    bleu3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    return bleu1, bleu2, bleu3, bleu4